In [98]:
import os
import numpy as np
import pandas as pd
import plotly.express as py
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

In [23]:
df = pd.read_csv('../input/samsung-electronics-stock-historical-price/005930.KS.csv')
df

,Date,Open,High,Low,Close,Adj Close,Volume
0,2019-06-03,42950.0,43900.0,42500.0,43800.0,40250.332031,15466580
1,2019-06-04,43400.0,43700.0,43000.0,43450.0,39928.699219,9913497
2,2019-06-05,44050.0,44200.0,43700.0,43900.0,40342.230469,12464135
3,2019-06-07,43600.0,44350.0,43450.0,44200.0,40617.910156,11683682
4,2019-06-10,44300.0,44850.0,44050.0,44800.0,41169.292969,8792182
...,...,...,...,...,...,...,...
680,2022-03-07,70000.0,70600.0,69900.0,70100.0,70100.000000,18617138
681,2022-03-08,68800.0,70000.0,68700.0,69500.0,69500.000000,15828269
682,2022-03-10,69500.0,69500.0,69500.0,69500.0,69500.000000,0
683,2022-03-11,70500.0,70700.0,69700.0,70000.0,70000.000000,15787655


In [24]:
fig = make_subplots(specs=[[{'secondary_y':True}]])
fig.add_trace(
    go.Scatter(x=df.Date,y=df.Close,name='Close')
)
fig.add_trace(
    go.Scatter(x=df.Date,y=df['Adj Close'],name='Adj Close')
)
fig.update_layout(xaxis_title='Date',
                  yaxis_title='Price'
                 )
fig.show()

In [25]:
def bolinger(data):
    data['MA20'] = data.Close.rolling(window=20).mean()
    data['std'] = data.Close.rolling(window=20).std()
    data['upper_20'] = data.MA20 + 2 * data['std']
    data['lower_20'] = data.MA20 - 2 * data['std']
    data.drop('std',axis=1,inplace=True)
    data.dropna(inplace=True)
    return data

In [71]:
665 % (len(df)-1)

0

In [29]:
df = bolinger(df)
df.ffill(inplace=True)
df.reset_index(drop=True,inplace=True)
df

,Date,Open,High,Low,Close,Adj Close,Volume,MA20,upper_20,lower_20
0,2019-07-01,47350.0,47400.0,46250.0,46600.0,43157.710938,11383522,44952.5,47046.972676,42858.027324
1,2019-07-02,46200.0,46900.0,45850.0,46250.0,42833.570313,8463073,45075.0,47172.241296,42977.758704
2,2019-07-03,45750.0,46350.0,45200.0,45400.0,42046.359375,9669368,45172.5,47128.186715,43216.813285
3,2019-07-04,45250.0,46200.0,45250.0,46000.0,42602.046875,6365573,45277.5,47169.999044,43385.000956
4,2019-07-05,45950.0,45950.0,45250.0,45650.0,42277.890625,7235395,45350.0,47178.718274,43521.281726
...,...,...,...,...,...,...,...,...,...,...
661,2022-03-07,70000.0,70600.0,69900.0,70100.0,70100.000000,18617138,73330.0,76119.529294,70540.470706
662,2022-03-08,68800.0,70000.0,68700.0,69500.0,69500.000000,15828269,73155.0,76428.803518,69881.196482
663,2022-03-10,69500.0,69500.0,69500.0,69500.0,69500.000000,0,72980.0,76640.083404,69319.916596
664,2022-03-11,70500.0,70700.0,69700.0,70000.0,70000.000000,15787655,72830.0,76724.990034,68935.009966


In [27]:
fig = make_subplots(specs=[[{'secondary_y':True}]])
fig.add_trace(
    go.Scatter(x=df.Date,y=df.Close,name='Close')
)
fig.add_trace(
    go.Scatter(x=df.Date,y=df.upper_20,name='Upper 20')
)
fig.add_trace(
    go.Scatter(x=df.Date,y=df.lower_20,name='Lower 20')
)
fig.update_layout(title='ploting bolinger band',
                  xaxis_title='Date',
                  yaxis_title='Price'
                 )
fig.show()

# Invest Strategy
* if Close meet upper band or similar my backtesting function ---> sell stock
* if Close meet lower band or similar my backtesting function ---> buy stock

In [33]:
import math

In [93]:
total_balance = 1000000
def backtesting(df,total_balance):
    buy_price = []
    buy_date = []
    sell_price = []
    sell_date = []

    long_earn = []

    num_bought = 0
    df_balance = []
    df_date = []
    balance_date = []
    trade_num = 0
    buy_num = -1
    sell_num = -1
    trade_num = -1
    position = 'none'
    for i in range(len(df.Close)):
        if total_balance <= 0:
            print('you lost all money...')
            break
        # 포지션이 없을때 진입 타이밍(볼린저 밴드)
        elif position == 'none':
            if df.Close[i] <= df['lower_20'][i] and abs(df.Close[i] - df['lower_20'][i]) <= 1000:
                position = 'long'
                buy_price.append(df.Close[i])
                buy_date.append(df.Date[i])
                if trade_num >= 0:
                    total_balance = df_balance[trade_num]
                num_bought = 0
                num_count = math.floor(total_balance / df.Close[i])
                num_bought += num_count
                remain = total_balance - (df.Close[i] * num_count)
                df_balance.append(total_balance)
                df_date.append(df.Date[i])
                print('----------------------------------------------------------------------------')
                print(f'Meet Lower Bolinger Band So Enter the position:{position}')
                print(f'{df.Date[i]}\tBuy price:{df.Close[i]}\tNum bought:{num_bought}')
                print(f'Reamin Balance:{remain}')
                print('----------------------------------------------------------------------------')
#                 temp_price = df.vwap[i]
#                 temp_time = df.time[i]
                trade_num += 1
                buy_num += 1
#             else:
#                 temp_price = df.vwap[i]
#                 temp_symbol = df.symbol[i]
#                 temp_time = df.time[i]
        elif position == 'long':
            if df.Close[i] >= df['upper_20'][i] and abs(df.Close[i] - df['upper_20'][i]) <= 1000:
                position = 'none'
                sell_price.append(df.Close[i])
                sell_date.append(df.Date[i])
                total_balance = remain
                earn = df.Close[i] - buy_price[buy_num]
                total_earn = (earn * num_bought) + (buy_price[buy_num] * num_bought)
                
                total_balance += total_earn
                df_balance.append(total_balance)
                df_date.append(df.Date[i])
                print('----------------------------------------------------------------------------')
                print(f'Meet Upper Bolinger Band So Sell the stock')
                print(f'{df.Date[i]}\tBuy price:{buy_price[buy_num]}\tSell price:{df.Close[i]}')
                print(f'Remain Balance:{total_earn}')
                print('----------------------------------------------------------------------------')
                sell_num += 1
                trade_num += 1
#                 temp_price = df.Close[i]
#                 temp_time = df.Date[i]
#             else:
        
#                 temp_price = df.Close[i]
#                 temp_time = df.Date[i]
        if i % (len(df) - 1) == 0:
            if position == 'long':
                sell_price.append(df.Close[i])
                sell_date.append(df.Date[i])
                total_balance = remain
                earn = df.Close[i] - buy_price[buy_num]
                total_earn = (earn * num_bought) + (buy_price[buy_num] * num_bought)
                total_balance += total_earn
                df_balance.append(total_balance)
                df_date.append(df.Date[i])
                print('----------------------------------------------------------------------------')
                print(f'End backtesing so sell the stock')
                print(f'{df.Date[i]}\tBuy price:{buy_price[buy_num]}\tSell price:{df.Close[i]}')
                print(f'Remain Balance:{total_earn}')
                print('----------------------------------------------------------------------------')
            elif position == 'none':
                print('End Backtesting')
    print(f'Total Trading Num:{trade_num}')
    balance = pd.DataFrame({'Date': df_date,'balance':df_balance })
    final_return_ratio = ((balance.iloc[-1][1] / 1000000) - 1) * 100
    print('Return Ratio:{:.3f}%'.format(final_return_ratio))
    print(i)
    return balance

In [94]:
df_balance = backtesting(df,total_balance)

End Backtesting
----------------------------------------------------------------------------
Meet Lower Bolinger Band So Enter the position:long
2019-08-05	Buy price:43950.0	Num bought:22
Reamin Balance:33100.0
----------------------------------------------------------------------------
----------------------------------------------------------------------------
Meet Upper Bolinger Band So Sell the stock
2019-09-05	Buy price:43950.0	Sell price:45700.0
Remain Balance:1005400.0
----------------------------------------------------------------------------
----------------------------------------------------------------------------
Meet Lower Bolinger Band So Enter the position:long
2019-11-29	Buy price:50300.0	Num bought:20
Reamin Balance:32500.0
----------------------------------------------------------------------------
----------------------------------------------------------------------------
Meet Upper Bolinger Band So Sell the stock
2019-12-13	Buy price:50300.0	Sell price:54700.0
Re

In [97]:
fig = make_subplots()
fig.add_trace(
    go.Scatter(x=df_balance.Date,y=df_balance.balance,name='your Balance')
)
fig.update_layout(
    xaxis_title='Date',
    yaxis_title='your Balance'
)
fig.show()